In [ ]:
%cd ..

import copy
import torch

from torch_geometric.datasets import FakeDataset

from nn.net_builder import CustomMlp, CustomGnn, RolandNet

In [ ]:
# Create a quick fake dataset to get data to test a GNN on
# COpying to ensure each "timestep" has the same number of nodes (current implementation wont work otherwise, can be fixed later through a padding and masking approach)
fds = FakeDataset(num_graphs=10, avg_num_nodes=10, avg_degree=2, num_channels=6, edge_dim=3)
fds_seq = [copy.deepcopy(fds.get(0)) for _ in range(3)]
fds_seq

[Data(y=[1], edge_index=[2, 36], x=[11, 6], edge_attr=[36, 3]),
 Data(y=[1], edge_index=[2, 36], x=[11, 6], edge_attr=[36, 3]),
 Data(y=[1], edge_index=[2, 36], x=[11, 6], edge_attr=[36, 3])]

In [3]:
# ROLAND with GCN Core

x_feats=6
x_out=3
core_hidden=16
core_blocks=2
encodings='rwpe'
encode_size=4

#Arguments for positional encodings, in this case RWPE
encode_args={'walk_length': encode_size}

#arguments for the Message Passing Blocks
mp_args = {'in_channels': core_hidden, 'out_channels': core_hidden}
gnn_norm_args = {'in_channels': core_hidden}
gnn_act_args = {}
gnn_edge_drop_args = {'p': 0.05, 'force_undirected': True}

mpblock_args = {'mp_type': 'gcn', 'mp_args': mp_args, 'norm_type': 'batch', 'norm_args': gnn_norm_args,
                'act_type': 'relu', 'act_args': gnn_act_args, 'drop_graph': 'edge', 'drop_graph_args': gnn_edge_drop_args,
                'drop_x': True, 'dropx_p': 0.1, 'residual_node': True, 'edge_attr': False}

#Arguments for the Feed Forward Blocks
ff_hidden = 32

ffblock_args = {'in_size': core_hidden, 'out_size': core_hidden, 'hidden_size': ff_hidden, 'norm_type': 'batch',
                'norm_args': {'num_features': core_hidden}, 'act_type': 'relu', 'act_args': {}}

#Combine into Arguments for the 'Core' of the GNN, includes size of the GRU
core_args = {'mp_args': mpblock_args, 'ff_args': ffblock_args, 'gru_in': core_hidden}

test_roland = RolandNet(x_feats=x_feats, x_out=x_out, core_hidden=core_hidden, core_blocks=core_blocks, core_args=core_args,
                     encodings=encodings, encode_size=encode_size, encode_args=encode_args)

In [4]:
test_roland.gnn_core

ModuleList(
  (0-1): 2 x RolandBlock(
    (message_pass): MessagePassBlock(
      <drop_x=(True, 0.1), residual=True>
      (drop_graph): GraphEdgeDropout(p=0.05, fu=True)
      (mp_layer): GCNConv(16, 16)
      (norm_layer): BatchNorm(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act_layer): ReLU()
    )
    (feed_forward): FeedForwardBlock(
      residual=True
      (lin1): Linear(16, 32, bias=True)
      (act_layer): ReLU()
      (lin2): Linear(32, 16, bias=True)
      (norm_layer): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (grucell): GRUCell(16, 16)
  )
)

In [ ]:
#initial hiddens: zero matrices serving as the initial hidden for the GRU Cell

hid_init = [torch.zeros((1, fds_seq[0].x.shape[0], core_hidden)) for n in range(core_blocks)]
hid_init = torch.concat(hid_init, dim=0)

In [6]:
outs, hiddens = test_roland(fds_seq, hid_init)
# Output: (S, N, F) Sequence Length, Batch Size, Output Feature Length
print(f"output shape: {outs.shape}")
# Hiddens: (S, L, N, H) Sequence Length, Model Depth, Batch Size, Hidden Length
print(f"hidden shape: {hiddens.shape}")

output shape: torch.Size([3, 11, 3])
hidden shape: torch.Size([3, 2, 11, 16])


In [7]:
outs[0]

tensor([[-0.2131,  0.0803, -0.4059],
        [-0.2641,  0.0265, -0.4152],
        [-0.3256,  0.0207, -0.3979],
        [-0.2288, -0.0265, -0.4524],
        [-0.4889, -0.0444, -0.3759],
        [-0.3298,  0.0339, -0.3901],
        [-0.2302, -0.0324, -0.4548],
        [-0.2049,  0.0806, -0.4084],
        [-0.4430, -0.0514, -0.3943],
        [-0.2053,  0.0788, -0.4091],
        [-0.2233, -0.0019, -0.4423]], grad_fn=<SelectBackward0>)